In [ ]:
!pip install -q kaggle torchinfo

In [ ]:
import os
import random
import time
import copy
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from torchvision import transforms, models

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    roc_curve,
    auc,
    precision_recall_curve,
    average_precision_score
)
from sklearn.preprocessing import label_binarize

from torchinfo import summary

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
from google.colab import files
files.upload()

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

print("Kaggle API configured.")

In [ ]:
!kaggle datasets download -d mohammadhossein77/brain-tumors-dataset
!unzip -q -o brain-tumors-dataset.zip -d brain_tumors_dataset

print("Dataset downloaded and extracted.")

In [ ]:
root = Path("/content/brain_tumors_dataset")

print("Root exists:", root.exists())
print("\nTop-level folders:")
for item in sorted(root.iterdir()):
    print(item)

In [ ]:
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

data_root = Path("/content/brain_tumors_dataset/Data")

class_folders = {
    "Glioma": data_root / "Tumor" / "glioma_tumor",
    "Meningioma": data_root / "Tumor" / "meningioma_tumor",
    "Pituitary": data_root / "Tumor" / "pituitary_tumor",
    "No Tumor": data_root / "Normal",
}

records = []

for label, folder in class_folders.items():
    print(label, "->", folder, "exists:", folder.exists())

    for img_path in folder.rglob("*"):
        if img_path.suffix.lower() in IMG_EXTS:
            records.append({
                "filepath": str(img_path),
                "label": label
            })

df = pd.DataFrame(records)
df = df.drop_duplicates(subset="filepath").reset_index(drop=True)

print("Total images:", len(df))
print(df["label"].value_counts())

df.head()

In [ ]:
class_counts = df["label"].value_counts().sort_index()

plt.figure(figsize=(8, 5))
plt.bar(class_counts.index, class_counts.values)
plt.title("Class Distribution")
plt.xlabel("Class")
plt.ylabel("Number of Images")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

In [ ]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.20,
    stratify=df["label"],
    random_state=SEED
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=SEED
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

print("\nTrain distribution:")
print(train_df["label"].value_counts())

print("\nValidation distribution:")
print(val_df["label"].value_counts())

print("\nTest distribution:")
print(test_df["label"].value_counts())

In [ ]:
summary_df = pd.concat([
    train_df["label"].value_counts().rename("Train"),
    val_df["label"].value_counts().rename("Validation"),
    test_df["label"].value_counts().rename("Test")
], axis=1).fillna(0)

summary_df.plot(kind="bar", figsize=(9, 5))
plt.title("Class Distribution Across Splits")
plt.xlabel("Class")
plt.ylabel("Number of Images")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

In [ ]:
CLASS_NAMES = ["Glioma", "Meningioma", "Pituitary", "No Tumor"]
NUM_CLASSES = len(CLASS_NAMES)

class_to_idx = {cls: i for i, cls in enumerate(CLASS_NAMES)}
idx_to_class = {i: cls for cls, i in class_to_idx.items()}

print(class_to_idx)

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

In [ ]:
class BrainTumorDataset(Dataset):
    def __init__(self, dataframe, transform=None, class_to_idx=None, return_path=False):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform
        self.class_to_idx = class_to_idx
        self.return_path = return_path

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        img_path = row["filepath"]
        label_name = row["label"]

        image = Image.open(img_path).convert("RGB")
        label = self.class_to_idx[label_name]

        if self.transform:
            image = self.transform(image)

        if self.return_path:
            return image, label, img_path

        return image, label

In [ ]:
BATCH_SIZE = 32
NUM_WORKERS = 2

train_dataset = BrainTumorDataset(train_df, train_transform, class_to_idx)
val_dataset = BrainTumorDataset(val_df, eval_transform, class_to_idx)
test_dataset = BrainTumorDataset(test_df, eval_transform, class_to_idx)
test_dataset_path = BrainTumorDataset(test_df, eval_transform, class_to_idx, return_path=True)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

test_loader_path = DataLoader(
    test_dataset_path,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

images, labels = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)

In [ ]:
def denormalize(img_tensor, mean=IMAGENET_MEAN, std=IMAGENET_STD):
    img = img_tensor.detach().clone().cpu()

    for t, m, s in zip(img, mean, std):
        t.mul_(s).add_(m)

    img = torch.clamp(img, 0, 1)
    return img


images, labels = next(iter(train_loader))

plt.figure(figsize=(12, 6))

for i in range(8):
    plt.subplot(2, 4, i + 1)
    img = denormalize(images[i]).permute(1, 2, 0).numpy()
    plt.imshow(img)
    plt.title(idx_to_class[int(labels[i])])
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
class ChannelAttention(nn.Module):
    def __init__(self, in_channels, reduction=16):
        super().__init__()

        hidden_channels = max(in_channels // reduction, 1)

        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)

        self.mlp = nn.Sequential(
            nn.Conv2d(in_channels, hidden_channels, kernel_size=1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden_channels, in_channels, kernel_size=1, bias=False)
        )

        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.mlp(self.avg_pool(x))
        max_out = self.mlp(self.max_pool(x))

        attention = self.sigmoid(avg_out + max_out)

        return x * attention


class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()

        padding = kernel_size // 2

        self.conv = nn.Conv2d(
            2,
            1,
            kernel_size=kernel_size,
            padding=padding,
            bias=False
        )

        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)

        attention = torch.cat([avg_out, max_out], dim=1)
        attention = self.sigmoid(self.conv(attention))

        return x * attention


class CBAM(nn.Module):
    def __init__(self, in_channels, reduction=16, spatial_kernel_size=7):
        super().__init__()

        self.channel_attention = ChannelAttention(in_channels, reduction)
        self.spatial_attention = SpatialAttention(spatial_kernel_size)

    def forward(self, x):
        x = self.channel_attention(x)
        x = self.spatial_attention(x)

        return x

In [ ]:
class MobileNetV2_CBAM(nn.Module):
    def __init__(self, num_classes=4, dropout=0.3):
        super().__init__()

        weights = models.MobileNet_V2_Weights.DEFAULT
        backbone = models.mobilenet_v2(weights=weights)

        self.features = backbone.features

        # MobileNetV2 final feature channels = 1280
        self.cbam = CBAM(in_channels=1280, reduction=16)

        self.pool = nn.AdaptiveAvgPool2d(1)

        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(1280, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.cbam(x)
        x = self.pool(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)

        return x

In [ ]:
model = MobileNetV2_CBAM(num_classes=NUM_CLASSES, dropout=0.3).to(device)

# Freeze early MobileNetV2 layers
for param in model.features.parameters():
    param.requires_grad = False

# Unfreeze last part
for param in model.features[14:].parameters():
    param.requires_grad = True

# CBAM and classifier should train
for param in model.cbam.parameters():
    param.requires_grad = True

for param in model.classifier.parameters():
    param.requires_grad = True


def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


total_params, trainable_params = count_params(model)

print("Total parameters:", total_params)
print("Trainable parameters:", trainable_params)

summary(model, input_size=(1, 3, 224, 224))

In [ ]:
def get_class_weights(train_df, class_names):
    counts = train_df["label"].value_counts().reindex(class_names).values.astype(np.float32)
    weights = len(train_df) / (len(class_names) * counts)

    return torch.tensor(weights, dtype=torch.float32)


class_weights = get_class_weights(train_df, CLASS_NAMES).to(device)

print("Class weights:", class_weights)

criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4,
    weight_decay=1e-4
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2
)

scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, scaler, device):
    model.train()

    running_loss = 0.0
    all_preds = []
    all_labels = []

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * images.size(0)

        preds = outputs.argmax(dim=1)

        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)

    return epoch_loss, epoch_acc

In [ ]:
@torch.no_grad()
def evaluate_model(model, loader, criterion, device):
    model.eval()

    running_loss = 0.0
    all_labels = []
    all_preds = []
    all_probs = []

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        probs = torch.softmax(outputs, dim=1)
        preds = outputs.argmax(dim=1)

        running_loss += loss.item() * images.size(0)

        all_labels.extend(labels.detach().cpu().numpy())
        all_preds.extend(preds.detach().cpu().numpy())
        all_probs.extend(probs.detach().cpu().numpy())

    all_labels = np.array(all_labels)
    all_preds = np.array(all_preds)
    all_probs = np.array(all_probs)

    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels,
        all_preds,
        average="weighted",
        zero_division=0
    )

    return {
        "loss": running_loss / len(loader.dataset),
        "accuracy": accuracy_score(all_labels, all_preds),
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "labels": all_labels,
        "preds": all_preds,
        "probs": all_probs
    }

In [ ]:
EPOCHS = 10
EARLY_STOP_PATIENCE = 4

history = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": [],
    "val_precision": [],
    "val_recall": [],
    "val_f1": [],
    "lr": []
}

best_wts = copy.deepcopy(model.state_dict())
best_val_f1 = -1
best_epoch = 0
epochs_without_improvement = 0

for epoch in range(EPOCHS):
    train_loss, train_acc = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        scaler,
        device
    )

    val_metrics = evaluate_model(
        model,
        val_loader,
        criterion,
        device
    )

    scheduler.step(val_metrics["loss"])

    current_lr = optimizer.param_groups[0]["lr"]

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_metrics["loss"])
    history["val_acc"].append(val_metrics["accuracy"])
    history["val_precision"].append(val_metrics["precision"])
    history["val_recall"].append(val_metrics["recall"])
    history["val_f1"].append(val_metrics["f1"])
    history["lr"].append(current_lr)

    print(
        f"Epoch [{epoch+1}/{EPOCHS}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_metrics['loss']:.4f} | "
        f"Val Acc: {val_metrics['accuracy']:.4f} | "
        f"Val Precision: {val_metrics['precision']:.4f} | "
        f"Val Recall: {val_metrics['recall']:.4f} | "
        f"Val F1: {val_metrics['f1']:.4f} | "
        f"LR: {current_lr:.6f}"
    )

    if val_metrics["f1"] > best_val_f1:
        best_val_f1 = val_metrics["f1"]
        best_epoch = epoch + 1
        best_wts = copy.deepcopy(model.state_dict())
        torch.save(best_wts, "/content/best_mobilenetv2_cbam.pth")
        epochs_without_improvement = 0
        print("Saved best model.")
    else:
        epochs_without_improvement += 1

        if epochs_without_improvement >= EARLY_STOP_PATIENCE:
            print("Early stopping triggered.")
            break

print("Best validation F1:", best_val_f1)
print("Best epoch:", best_epoch)

In [ ]:
history_df = pd.DataFrame(history)
history_df.to_csv("/content/mobilenetv2_cbam_training_history.csv", index=False)

epochs_range = range(1, len(history_df) + 1)

plt.figure(figsize=(7, 4))
plt.plot(epochs_range, history_df["train_loss"], marker="o", label="Train Loss")
plt.plot(epochs_range, history_df["val_loss"], marker="o", label="Validation Loss")
plt.title("MobileNetV2 + CBAM Loss Curve")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(epochs_range, history_df["train_acc"], marker="o", label="Train Accuracy")
plt.plot(epochs_range, history_df["val_acc"], marker="o", label="Validation Accuracy")
plt.title("MobileNetV2 + CBAM Accuracy Curve")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(epochs_range, history_df["val_precision"], marker="o", label="Validation Precision")
plt.plot(epochs_range, history_df["val_recall"], marker="o", label="Validation Recall")
plt.plot(epochs_range, history_df["val_f1"], marker="o", label="Validation F1")
plt.title("Validation Precision, Recall, F1")
plt.xlabel("Epoch")
plt.ylabel("Score")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(epochs_range, history_df["lr"], marker="o")
plt.title("Learning Rate Curve")
plt.xlabel("Epoch")
plt.ylabel("Learning Rate")
plt.tight_layout()
plt.show()

In [ ]:
model.load_state_dict(torch.load("/content/best_mobilenetv2_cbam.pth", map_location=device))
model = model.to(device)

test_metrics = evaluate_model(model, test_loader, criterion, device)

print("MobileNetV2 + CBAM Test Results")
print("Test Loss      :", round(test_metrics["loss"], 4))
print("Test Accuracy  :", round(test_metrics["accuracy"], 4))
print("Test Precision :", round(test_metrics["precision"], 4))
print("Test Recall    :", round(test_metrics["recall"], 4))
print("Test F1-score  :", round(test_metrics["f1"], 4))

In [ ]:
report = classification_report(
    test_metrics["labels"],
    test_metrics["preds"],
    target_names=CLASS_NAMES,
    digits=4,
    zero_division=0
)

print(report)

In [ ]:
report_dict = classification_report(
    test_metrics["labels"],
    test_metrics["preds"],
    target_names=CLASS_NAMES,
    digits=4,
    zero_division=0,
    output_dict=True
)

report_df = pd.DataFrame(report_dict).T
report_df.to_csv("/content/mobilenetv2_cbam_classification_report.csv")

report_df

In [ ]:
cm = confusion_matrix(test_metrics["labels"], test_metrics["preds"])

plt.figure(figsize=(7, 6))
plt.imshow(cm)
plt.title("MobileNetV2 + CBAM Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.xticks(np.arange(NUM_CLASSES), CLASS_NAMES, rotation=30, ha="right")
plt.yticks(np.arange(NUM_CLASSES), CLASS_NAMES)

for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        plt.text(j, i, cm[i, j], ha="center", va="center")

plt.colorbar()
plt.tight_layout()
plt.show()

In [ ]:
cm_norm = cm.astype("float") / cm.sum(axis=1, keepdims=True)
cm_norm = np.nan_to_num(cm_norm)

plt.figure(figsize=(7, 6))
plt.imshow(cm_norm, vmin=0, vmax=1)
plt.title("Normalized Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.xticks(np.arange(NUM_CLASSES), CLASS_NAMES, rotation=30, ha="right")
plt.yticks(np.arange(NUM_CLASSES), CLASS_NAMES)

for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        plt.text(j, i, f"{cm_norm[i, j]:.2f}", ha="center", va="center")

plt.colorbar()
plt.tight_layout()
plt.show()

In [ ]:
per_class_df = report_df.loc[CLASS_NAMES, ["precision", "recall", "f1-score", "support"]]
per_class_df.to_csv("/content/mobilenetv2_cbam_per_class_metrics.csv")

per_class_df[["precision", "recall", "f1-score"]].plot(
    kind="bar",
    figsize=(10, 5)
)

plt.title("Per-Class Precision, Recall, F1-score")
plt.xlabel("Class")
plt.ylabel("Score")
plt.ylim(0, 1.05)
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

per_class_df

In [ ]:
y_true = test_metrics["labels"]
y_score = test_metrics["probs"]

y_true_bin = label_binarize(y_true, classes=list(range(NUM_CLASSES)))

plt.figure(figsize=(8, 6))

roc_results = []

for i, class_name in enumerate(CLASS_NAMES):
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_score[:, i])
    roc_auc = auc(fpr, tpr)

    roc_results.append({
        "class": class_name,
        "roc_auc": roc_auc
    })

    plt.plot(fpr, tpr, label=f"{class_name} AUC={roc_auc:.3f}")

plt.plot([0, 1], [0, 1], linestyle="--", label="Chance")
plt.title("MobileNetV2 + CBAM ROC Curves")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.tight_layout()
plt.show()

roc_df = pd.DataFrame(roc_results)
roc_df.to_csv("/content/mobilenetv2_cbam_roc_auc.csv", index=False)

roc_df

In [ ]:
plt.figure(figsize=(8, 6))

pr_results = []

for i, class_name in enumerate(CLASS_NAMES):
    precision, recall, _ = precision_recall_curve(y_true_bin[:, i], y_score[:, i])
    ap = average_precision_score(y_true_bin[:, i], y_score[:, i])

    pr_results.append({
        "class": class_name,
        "average_precision": ap
    })

    plt.plot(recall, precision, label=f"{class_name} AP={ap:.3f}")

plt.title("MobileNetV2 + CBAM Precision-Recall Curves")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.legend()
plt.tight_layout()
plt.show()

pr_df = pd.DataFrame(pr_results)
pr_df.to_csv("/content/mobilenetv2_cbam_average_precision.csv", index=False)

pr_df

In [ ]:
def measure_inference_time(model, loader, device):
    model.eval()

    total_images = 0

    if device.type == "cuda":
        torch.cuda.synchronize()

    start = time.perf_counter()

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            _ = model(images)
            total_images += images.size(0)

    if device.type == "cuda":
        torch.cuda.synchronize()

    end = time.perf_counter()

    total_time = end - start
    avg_time = total_time / total_images

    return total_time, avg_time


total_time, avg_time = measure_inference_time(model, test_loader, device)

torch.save(model.state_dict(), "/content/mobilenetv2_cbam_final.pth")

model_size_mb = os.path.getsize("/content/mobilenetv2_cbam_final.pth") / (1024 ** 2)

total_params, trainable_params = count_params(model)

result_summary = pd.DataFrame([{
    "model": "MobileNetV2 + CBAM",
    "test_accuracy": test_metrics["accuracy"],
    "test_precision": test_metrics["precision"],
    "test_recall": test_metrics["recall"],
    "test_f1": test_metrics["f1"],
    "total_parameters": total_params,
    "trainable_parameters": trainable_params,
    "model_size_mb": model_size_mb,
    "total_inference_time_sec": total_time,
    "avg_inference_time_per_image_sec": avg_time,
    "best_val_f1": best_val_f1,
    "best_epoch": best_epoch
}])

result_summary.to_csv("/content/mobilenetv2_cbam_result_summary.csv", index=False)

result_summary

In [ ]:
metric_names = ["Accuracy", "Precision", "Recall", "F1-score"]
metric_values = [
    test_metrics["accuracy"],
    test_metrics["precision"],
    test_metrics["recall"],
    test_metrics["f1"]
]

plt.figure(figsize=(7, 4))
plt.bar(metric_names, metric_values)
plt.title("MobileNetV2 + CBAM Final Test Metrics")
plt.ylabel("Score")
plt.ylim(0, 1.05)

for i, v in enumerate(metric_values):
    plt.text(i, v + 0.01, f"{v:.3f}", ha="center")

plt.tight_layout()
plt.show()

In [ ]:
prediction_df = test_df.copy()

prediction_df["true_idx"] = test_metrics["labels"]
prediction_df["pred_idx"] = test_metrics["preds"]

prediction_df["true_label"] = prediction_df["true_idx"].map(idx_to_class)
prediction_df["pred_label"] = prediction_df["pred_idx"].map(idx_to_class)

prediction_df["confidence"] = test_metrics["probs"].max(axis=1)
prediction_df["correct"] = prediction_df["true_idx"] == prediction_df["pred_idx"]

for i, class_name in enumerate(CLASS_NAMES):
    prediction_df[f"prob_{class_name}"] = test_metrics["probs"][:, i]

prediction_df.to_csv("/content/mobilenetv2_cbam_test_predictions.csv", index=False)

prediction_df.head()

In [ ]:
show_df = prediction_df.sample(min(12, len(prediction_df)), random_state=SEED).reset_index(drop=True)

plt.figure(figsize=(14, 10))

for i, row in show_df.iterrows():
    image = Image.open(row["filepath"]).convert("RGB").resize((224, 224))

    plt.subplot(3, 4, i + 1)
    plt.imshow(image)

    title = f"True: {row['true_label']}\nPred: {row['pred_label']}\nConf: {row['confidence']:.2f}"
    plt.title(title, fontsize=9)
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
wrong_df = prediction_df[prediction_df["correct"] == False].sort_values(
    "confidence",
    ascending=False
)

print("Wrong predictions:", len(wrong_df))

if len(wrong_df) > 0:
    show_wrong = wrong_df.head(12).reset_index(drop=True)

    plt.figure(figsize=(14, 10))

    for i, row in show_wrong.iterrows():
        image = Image.open(row["filepath"]).convert("RGB").resize((224, 224))

        plt.subplot(3, 4, i + 1)
        plt.imshow(image)

        title = f"True: {row['true_label']}\nPred: {row['pred_label']}\nConf: {row['confidence']:.2f}"
        plt.title(title, fontsize=9)
        plt.axis("off")

    plt.tight_layout()
    plt.show()
else:
    print("No wrong predictions found.")

In [ ]:
class SimpleGradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer

        self.activations = None
        self.gradients = None

        self.forward_hook = target_layer.register_forward_hook(self.save_activation)
        self.backward_hook = target_layer.register_full_backward_hook(self.save_gradient)

    def save_activation(self, module, input, output):
        self.activations = output.detach()

    def save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def remove_hooks(self):
        self.forward_hook.remove()
        self.backward_hook.remove()

    def __call__(self, input_tensor, class_idx=None):
        self.model.eval()
        self.model.zero_grad()

        output = self.model(input_tensor)

        pred_idx = int(output.argmax(dim=1).item())

        if class_idx is None:
            class_idx = pred_idx

        score = output[:, class_idx].sum()
        score.backward()

        gradients = self.gradients
        activations = self.activations

        weights = gradients.mean(dim=(2, 3), keepdim=True)

        cam = (weights * activations).sum(dim=1, keepdim=True)
        cam = F.relu(cam)

        cam = F.interpolate(
            cam,
            size=input_tensor.shape[2:],
            mode="bilinear",
            align_corners=False
        )

        cam = cam.squeeze().detach().cpu().numpy()

        cam = cam - cam.min()
        cam = cam / (cam.max() + 1e-8)

        return cam, pred_idx

In [ ]:
def tensor_to_np_image(img_tensor):
    img = denormalize(img_tensor)
    img = img.permute(1, 2, 0).cpu().numpy()
    return img


def plot_gradcam(row, gradcam):
    image = Image.open(row["filepath"]).convert("RGB")

    tensor = eval_transform(image).unsqueeze(0).to(device)

    cam, pred_idx = gradcam(tensor, class_idx=int(row["pred_idx"]))

    img_np = tensor_to_np_image(tensor.squeeze(0))

    plt.figure(figsize=(12, 4))

    plt.subplot(1, 3, 1)
    plt.imshow(img_np)
    plt.title(f"Original\nTrue: {row['true_label']}")
    plt.axis("off")

    plt.subplot(1, 3, 2)
    plt.imshow(cam, cmap="jet")
    plt.title("Grad-CAM Heatmap")
    plt.axis("off")

    plt.subplot(1, 3, 3)
    plt.imshow(img_np)
    plt.imshow(cam, cmap="jet", alpha=0.45)
    plt.title(f"Overlay\nPred: {row['pred_label']}")
    plt.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
target_layer = model.features[-1]
gradcam = SimpleGradCAM(model, target_layer)

selected_rows = []

for class_name in CLASS_NAMES:
    subset = prediction_df[
        (prediction_df["true_label"] == class_name) &
        (prediction_df["correct"] == True)
    ]

    if len(subset) == 0:
        subset = prediction_df[prediction_df["true_label"] == class_name]

    if len(subset) > 0:
        selected_rows.append(
            subset.sort_values("confidence", ascending=False).iloc[0]
        )

for row in selected_rows:
    plot_gradcam(row, gradcam)

gradcam.remove_hooks()

In [ ]:
from google.colab import files

files.download("/content/best_mobilenetv2_cbam.pth")
files.download("/content/mobilenetv2_cbam_final.pth")
files.download("/content/mobilenetv2_cbam_result_summary.csv")
files.download("/content/mobilenetv2_cbam_classification_report.csv")
files.download("/content/mobilenetv2_cbam_training_history.csv")
files.download("/content/mobilenetv2_cbam_test_predictions.csv")